# 05 — Download dos Resultados do Experimento

Este notebook documenta a etapa de empacotamento e download dos artefatos gerados no RunPod.

A ideia desta etapa é preparar tudo que será necessário para continuar a análise localmente, sem depender mais da GPU.

Este notebook deve ser executado depois que as etapas anteriores já tiverem sido concluídas:

* `01_environment_setup.ipynb`;
* `02_data_preparation.ipynb`;
* `03_training_documentation.ipynb`;
* `04_evaluation_and_raw_outputs.ipynb`.

Ao final desta etapa, será gerado um arquivo `.tar.gz` contendo os principais artefatos do experimento, incluindo:

* notebooks;
* scripts;
* dados processados;
* adapters treinados;
* logs;
* outputs brutos;
* métricas;
* manifestos de execução.

Arquivos pesados e desnecessários para a análise local, como `.venv`, cache do Hugging Face e checkpoints do modelo base, não serão incluídos no pacote.

## 1. Quando a GPU deixa de ser necessária?

A GPU é necessária para as etapas que executam o modelo Llama, principalmente:

* treinamento dos adapters C2, C3 e C4;
* avaliação dos cenários C0, C1, C2, C3 e C4;
* geração dos outputs brutos.

Depois que os arquivos `.jsonl` de outputs brutos e os arquivos `.csv` de métricas forem gerados, a GPU não é mais necessária para as etapas finais de análise.

A partir desse ponto, é possível baixar os resultados e continuar localmente com notebooks de análise, geração de tabelas, gráficos e escrita dos resultados.

Antes de desligar o RunPod, é importante confirmar que os arquivos de avaliação existem.

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import os
import subprocess
import sys

import pandas as pd


print("Imports carregados.")

Imports carregados.


## 2. Configuração dos diretórios

Nesta etapa são definidos os caminhos principais do projeto e da execução experimental.

O pacote final será salvo em:

```text
/workspace/pi-defense-exp/exports/
```

A pasta `exports/` será usada apenas para armazenar os arquivos compactados que serão baixados para a máquina local.

In [2]:
PROJECT_DIR = Path("/workspace/pi-defense-exp")
os.chdir(PROJECT_DIR)

EXPERIMENT_NAME = "opi_sst2_sms_5k_872"
DATASET_SEED = 42
TRAINING_SEEDS = [42, 123, 2026]

RUN_ROOT = PROJECT_DIR / "runs" / EXPERIMENT_NAME / f"dataset_seed{DATASET_SEED}"
BASELINE_ROOT = RUN_ROOT / "baselines" / "c0_c1"

EXPORTS_DIR = PROJECT_DIR / "exports"
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_NAME = "pi-defense-exp-artifacts.tar.gz"
ARCHIVE_PATH = EXPORTS_DIR / ARCHIVE_NAME

HASH_NAME = "pi-defense-exp-artifacts.sha256"
HASH_PATH = EXPORTS_DIR / HASH_NAME

print("PROJECT_DIR:", PROJECT_DIR)
print("RUN_ROOT:", RUN_ROOT)
print("EXPORTS_DIR:", EXPORTS_DIR)
print("ARCHIVE_PATH:", ARCHIVE_PATH)
print("HASH_PATH:", HASH_PATH)

PROJECT_DIR: /workspace/pi-defense-exp
RUN_ROOT: /workspace/pi-defense-exp/runs/opi_sst2_sms_5k_872/dataset_seed42
EXPORTS_DIR: /workspace/pi-defense-exp/exports
ARCHIVE_PATH: /workspace/pi-defense-exp/exports/pi-defense-exp-artifacts.tar.gz
HASH_PATH: /workspace/pi-defense-exp/exports/pi-defense-exp-artifacts.sha256


## 3. Verificação dos principais artefatos

Antes de compactar os resultados, esta seção verifica se os principais arquivos esperados existem.

São verificados:

* dados processados;
* scripts;
* notebooks;
* pasta da execução;
* outputs brutos de C0/C1;
* métricas de C0/C1;
* outputs brutos de C2/C3/C4;
* métricas de C2/C3/C4;
* adapters treinados.

Essa verificação ajuda a evitar baixar um pacote incompleto.

In [3]:
def count_jsonl_lines(path: Path) -> int:
    if not path.exists():
        return 0

    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def file_size_mb(path: Path) -> float:
    if not path.exists() or not path.is_file():
        return 0.0

    return round(path.stat().st_size / 1024**2, 3)


def directory_size_mb(path: Path) -> float:
    if not path.exists() or not path.is_dir():
        return 0.0

    total = 0

    for file in path.rglob("*"):
        if file.is_file():
            total += file.stat().st_size

    return round(total / 1024**2, 3)


def seed_root(seed: int) -> Path:
    return RUN_ROOT / f"train_seed{seed}"


def adapter_dir_for(seed: int, scenario: str) -> Path:
    mapping = {
        "C2": seed_root(seed) / "adapters" / "c2_struq_sft",
        "C3": seed_root(seed) / "adapters" / "c3_secalign_dpo",
        "C4": seed_root(seed) / "adapters" / "c4_instruction_hierarchy_sft",
    }
    return mapping[scenario]


def raw_path_for(seed: int, scenario: str) -> Path:
    mapping = {
        "C2": seed_root(seed) / "raw" / "c2_struq_sft_raw_generations.jsonl",
        "C3": seed_root(seed) / "raw" / "c3_secalign_dpo_raw_generations.jsonl",
        "C4": seed_root(seed) / "raw" / "c4_instruction_hierarchy_sft_raw_generations.jsonl",
    }
    return mapping[scenario]


def metrics_path_for(seed: int, scenario: str) -> Path:
    mapping = {
        "C2": seed_root(seed) / "metrics" / "c2_struq_sft_metrics.csv",
        "C3": seed_root(seed) / "metrics" / "c3_secalign_dpo_metrics.csv",
        "C4": seed_root(seed) / "metrics" / "c4_instruction_hierarchy_sft_metrics.csv",
    }
    return mapping[scenario]


def baseline_raw_path() -> Path:
    return BASELINE_ROOT / "raw" / "c0_c1_raw_generations.jsonl"


def baseline_metrics_path() -> Path:
    return BASELINE_ROOT / "metrics" / "c0_c1_metrics.csv"

In [4]:
artifact_rows = []

basic_paths = [
    ("notebooks", PROJECT_DIR / "notebooks"),
    ("scripts", PROJECT_DIR / "scripts"),
    ("data_processed", PROJECT_DIR / "data" / "processed"),
    ("run_root", RUN_ROOT),
]

for name, path in basic_paths:
    artifact_rows.append({
        "group": "basic",
        "name": name,
        "path": str(path),
        "exists": path.exists(),
        "type": "directory" if path.is_dir() else "file",
        "size_mb": directory_size_mb(path) if path.is_dir() else file_size_mb(path),
        "lines": None,
    })

artifact_rows.append({
    "group": "baseline",
    "name": "c0_c1_raw",
    "path": str(baseline_raw_path()),
    "exists": baseline_raw_path().exists(),
    "type": "file",
    "size_mb": file_size_mb(baseline_raw_path()),
    "lines": count_jsonl_lines(baseline_raw_path()),
})

artifact_rows.append({
    "group": "baseline",
    "name": "c0_c1_metrics",
    "path": str(baseline_metrics_path()),
    "exists": baseline_metrics_path().exists(),
    "type": "file",
    "size_mb": file_size_mb(baseline_metrics_path()),
    "lines": None,
})

for seed in TRAINING_SEEDS:
    for scenario in ["C2", "C3", "C4"]:
        adapter_dir = adapter_dir_for(seed, scenario)
        raw_path = raw_path_for(seed, scenario)
        metrics_path = metrics_path_for(seed, scenario)

        artifact_rows.append({
            "group": "adapter",
            "name": f"{scenario}_seed{seed}_adapter",
            "path": str(adapter_dir),
            "exists": adapter_dir.exists(),
            "type": "directory",
            "size_mb": directory_size_mb(adapter_dir),
            "lines": None,
        })

        artifact_rows.append({
            "group": "raw",
            "name": f"{scenario}_seed{seed}_raw",
            "path": str(raw_path),
            "exists": raw_path.exists(),
            "type": "file",
            "size_mb": file_size_mb(raw_path),
            "lines": count_jsonl_lines(raw_path),
        })

        artifact_rows.append({
            "group": "metrics",
            "name": f"{scenario}_seed{seed}_metrics",
            "path": str(metrics_path),
            "exists": metrics_path.exists(),
            "type": "file",
            "size_mb": file_size_mb(metrics_path),
            "lines": None,
        })

artifact_check_df = pd.DataFrame(artifact_rows)
artifact_check_df

,group,name,path,exists,type,size_mb,lines
0,basic,notebooks,/workspace/pi-defense-exp/notebooks,True,directory,1.384,NaN
1,basic,scripts,/workspace/pi-defense-exp/scripts,True,directory,0.073,NaN
2,basic,data_processed,/workspace/pi-defense-exp/data/processed,True,directory,31.738,NaN
3,basic,run_root,/workspace/pi-defense-exp/runs/opi_sst2_sms_5k...,True,directory,6108.735,NaN
4,baseline,c0_c1_raw,/workspace/pi-defense-exp/runs/opi_sst2_sms_5k...,True,file,1.919,2616.0
5,baseline,c0_c1_metrics,/workspace/pi-defense-exp/runs/opi_sst2_sms_5k...,True,file,0.001,NaN
6,adapter,C2_seed42_adapter,/workspace/pi-defense-exp/runs/opi_sst2_sms_5k...,True,directory,673.352,NaN
7,raw,C2_seed42_raw,/workspace/pi-defense-exp/runs/opi_sst2_sms_5k...,True,file,1.450,1744.0
8,metrics,C2_seed42_metrics,/workspace/pi-defense-exp/runs/opi_sst2_sms_5k...,True,file,0.001,NaN
9,adapter,C3_seed42_adapter,/workspace/pi-defense-exp/runs/opi_sst2_sms_5k...,True,directory,673.383,NaN


## 4. Checagem de segurança antes de desligar o RunPod

Esta célula destaca arquivos ausentes.

Nem sempre todos os arquivos precisam existir. Por exemplo, se apenas a seed 42 foi executada, é esperado que os resultados das seeds 123 e 2026 ainda estejam ausentes.

Porém, antes de desligar o RunPod, é importante garantir que todos os resultados que você deseja analisar localmente já foram gerados.

In [5]:
missing_artifacts_df = artifact_check_df[artifact_check_df["exists"] == False].copy()

if len(missing_artifacts_df) == 0:
    print("Nenhum artefato ausente.")
else:
    print("Artefatos ausentes encontrados:")

missing_artifacts_df

Nenhum artefato ausente.


,group,name,path,exists,type,size_mb,lines


## 5. Manifesto local dos artefatos

Antes da compactação, será salvo um manifesto com a lista dos principais artefatos encontrados.

Esse arquivo ajuda a documentar o estado do experimento no momento do download.

O manifesto será salvo em:

```text
runs/opi_sst2_sms_5k_872/dataset_seed42/download_manifest.json
```

In [6]:
download_manifest = {
    "experiment_name": EXPERIMENT_NAME,
    "dataset_seed": DATASET_SEED,
    "training_seeds": TRAINING_SEEDS,
    "created_at": datetime.now().isoformat(),
    "project_dir": str(PROJECT_DIR),
    "run_root": str(RUN_ROOT),
    "exports_dir": str(EXPORTS_DIR),
    "archive_path": str(ARCHIVE_PATH),
    "hash_path": str(HASH_PATH),
    "artifacts": artifact_check_df.to_dict(orient="records"),
}

download_manifest_path = RUN_ROOT / "download_manifest.json"
download_manifest_path.parent.mkdir(parents=True, exist_ok=True)

with download_manifest_path.open("w", encoding="utf-8") as f:
    json.dump(download_manifest, f, ensure_ascii=False, indent=2)

print("Manifesto de download salvo em:", download_manifest_path)

Manifesto de download salvo em: /workspace/pi-defense-exp/runs/opi_sst2_sms_5k_872/dataset_seed42/download_manifest.json


## 6. Compactação dos artefatos

Nesta etapa será criado um arquivo `.tar.gz` com os principais artefatos do experimento.

O pacote inclui:

* `notebooks/`;
* `scripts/`;
* `data/processed/`;
* `runs/`.

O pacote não inclui:

* `.venv`;
* cache do Hugging Face;
* checkpoints globais do modelo base;
* arquivos temporários do Jupyter;
* diretórios `__pycache__`.

Essas exclusões reduzem bastante o tamanho do arquivo final e evitam baixar dependências ou pesos de modelo desnecessários.

In [7]:
CREATE_ARCHIVE = True

print("CREATE_ARCHIVE:", CREATE_ARCHIVE)

CREATE_ARCHIVE: True


In [8]:
archive_command = [
    "tar",
    "--exclude=.venv",
    "--exclude=cache",
    "--exclude=__pycache__",
    "--exclude=.ipynb_checkpoints",
    "-czvf",
    str(ARCHIVE_PATH),
    "notebooks",
    "scripts",
    "data/processed",
    "runs",
]

print("Comando de compactação:")
print(" ".join(archive_command))

Comando de compactação:
tar --exclude=.venv --exclude=cache --exclude=__pycache__ --exclude=.ipynb_checkpoints -czvf /workspace/pi-defense-exp/exports/pi-defense-exp-artifacts.tar.gz notebooks scripts data/processed runs


In [9]:
if CREATE_ARCHIVE:
    result = subprocess.run(
        archive_command,
        cwd=str(PROJECT_DIR),
        capture_output=True,
        text=True,
    )

    print("returncode:", result.returncode)

    if result.stdout:
        print("STDOUT:")
        print(result.stdout[-4000:])

    if result.stderr:
        print("STDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError("Erro ao criar o arquivo compactado.")
else:
    print("CREATE_ARCHIVE=False. A compactação não foi executada.")

returncode: 0
STDOUT:
t-625/
runs/opi_sst2_sms_5k_872/dataset_seed42/train_seed2026/adapters/c4_instruction_hierarchy_sft/checkpoint-625/README.md
runs/opi_sst2_sms_5k_872/dataset_seed42/train_seed2026/adapters/c4_instruction_hierarchy_sft/checkpoint-625/adapter_model.safetensors
runs/opi_sst2_sms_5k_872/dataset_seed42/train_seed2026/adapters/c4_instruction_hierarchy_sft/checkpoint-625/adapter_config.json
runs/opi_sst2_sms_5k_872/dataset_seed42/train_seed2026/adapters/c4_instruction_hierarchy_sft/checkpoint-625/chat_template.jinja
runs/opi_sst2_sms_5k_872/dataset_seed42/train_seed2026/adapters/c4_instruction_hierarchy_sft/checkpoint-625/tokenizer_config.json
runs/opi_sst2_sms_5k_872/dataset_seed42/train_seed2026/adapters/c4_instruction_hierarchy_sft/checkpoint-625/tokenizer.json
runs/opi_sst2_sms_5k_872/dataset_seed42/train_seed2026/adapters/c4_instruction_hierarchy_sft/checkpoint-625/training_args.bin
runs/opi_sst2_sms_5k_872/dataset_seed42/train_seed2026/adapters/c4_instruction_hiera

## 7. Geração do hash SHA-256

Após gerar o arquivo compactado, será criado um hash SHA-256.

Esse hash permite verificar depois se o arquivo baixado para a máquina local está íntegro.

O arquivo de hash será salvo em:

```text
exports/pi-defense-exp-artifacts.sha256
```

In [10]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    sha256 = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


if ARCHIVE_PATH.exists():
    archive_hash = sha256_file(ARCHIVE_PATH)

    with HASH_PATH.open("w", encoding="utf-8") as f:
        f.write(f"{archive_hash}  {ARCHIVE_NAME}\n")

    print("SHA-256:", archive_hash)
    print("Arquivo de hash salvo em:", HASH_PATH)
else:
    raise FileNotFoundError(f"Arquivo compactado não encontrado: {ARCHIVE_PATH}")

SHA-256: 98a28de699bbc9cd1dc20cc84f13d5cbdf4f139a959bfb02f70b085b4d14c079
Arquivo de hash salvo em: /workspace/pi-defense-exp/exports/pi-defense-exp-artifacts.sha256


## 8. Verificação do pacote final

Esta seção mostra o tamanho final do arquivo compactado e confirma que o `.tar.gz` e o `.sha256` foram criados.

Esses são os dois arquivos que devem ser baixados para a máquina local.

In [11]:
export_files_df = pd.DataFrame([
    {
        "file": ARCHIVE_PATH.name,
        "path": str(ARCHIVE_PATH),
        "exists": ARCHIVE_PATH.exists(),
        "size_mb": file_size_mb(ARCHIVE_PATH),
    },
    {
        "file": HASH_PATH.name,
        "path": str(HASH_PATH),
        "exists": HASH_PATH.exists(),
        "size_mb": file_size_mb(HASH_PATH),
    },
])

export_files_df

,file,path,exists,size_mb
0,pi-defense-exp-artifacts.tar.gz,/workspace/pi-defense-exp/exports/pi-defense-e...,True,5350.439
1,pi-defense-exp-artifacts.sha256,/workspace/pi-defense-exp/exports/pi-defense-e...,True,0.000


## 9. Comandos para baixar para a máquina local

Os comandos abaixo devem ser executados na sua máquina local, e não dentro do RunPod.

No RunPod, obtenha o IP e a porta SSH na tela de conexão do Pod.

Substitua:

```text
<IP>
<PORTA>
```

pelos dados mostrados no RunPod.

### Windows PowerShell

```powershell
scp -P <PORTA> root@<IP>:/workspace/pi-defense-exp/exports/pi-defense-exp-artifacts.tar.gz $env:USERPROFILE\Downloads\
scp -P <PORTA> root@<IP>:/workspace/pi-defense-exp/exports/pi-defense-exp-artifacts.sha256 $env:USERPROFILE\Downloads\
```

### Windows PowerShell com chave SSH

```powershell
scp -i $env:USERPROFILE\.ssh\id_ed25519 -P <PORTA> root@<IP>:/workspace/pi-defense-exp/exports/pi-defense-exp-artifacts.tar.gz $env:USERPROFILE\Downloads\
scp -i $env:USERPROFILE\.ssh\id_ed25519 -P <PORTA> root@<IP>:/workspace/pi-defense-exp/exports/pi-defense-exp-artifacts.sha256 $env:USERPROFILE\Downloads\
```

### Linux, macOS ou WSL

```bash
scp -P <PORTA> root@<IP>:/workspace/pi-defense-exp/exports/pi-defense-exp-artifacts.tar.gz .
scp -P <PORTA> root@<IP>:/workspace/pi-defense-exp/exports/pi-defense-exp-artifacts.sha256 .
```

## 10. Extração local dos resultados

Depois de baixar o arquivo `.tar.gz`, extraia os resultados na sua máquina local.

### Linux, macOS ou WSL

```bash
mkdir -p pi-defense-exp-local
tar -xzvf pi-defense-exp-artifacts.tar.gz -C pi-defense-exp-local
```

### Windows

No Windows, o arquivo pode ser extraído com:

* 7-Zip;
* WinRAR;
* Explorador de Arquivos;
* PowerShell com suporte a `tar`.

Exemplo no PowerShell:

```powershell
mkdir $env:USERPROFILE\Downloads\pi-defense-exp-local
tar -xzvf $env:USERPROFILE\Downloads\pi-defense-exp-artifacts.tar.gz -C $env:USERPROFILE\Downloads\pi-defense-exp-local
```

## 11. Verificação local do hash

Depois do download, é possível verificar se o arquivo chegou íntegro comparando o hash.

### Linux, macOS ou WSL

```bash
sha256sum -c pi-defense-exp-artifacts.sha256
```

### Windows PowerShell

No PowerShell, é possível calcular o hash com:

```powershell
Get-FileHash $env:USERPROFILE\Downloads\pi-defense-exp-artifacts.tar.gz -Algorithm SHA256
```

Depois, compare o valor retornado com o conteúdo do arquivo:

```powershell
type $env:USERPROFILE\Downloads\pi-defense-exp-artifacts.sha256
```

## 12. O que pode ser feito localmente depois do download?

Após baixar e extrair os artefatos, as próximas etapas podem ser feitas localmente sem GPU.

Isso inclui:

* análise das métricas;
* comparação entre cenários;
* comparação entre seeds;
* inspeção dos outputs brutos;
* geração de tabelas finais;
* geração de gráficos;
* escrita da seção de resultados;
* organização dos artefatos finais de reprodutibilidade.

As próximas etapas podem ser organizadas em notebooks como:

```text
06_metrics_analysis_and_comparison.ipynb
07_reproducibility_and_artifact_export.ipynb
```

A GPU só voltará a ser necessária se for preciso reexecutar o modelo, refazer treinamentos ou gerar novos outputs brutos.